In [1]:
import tkinter as tk
from tkinter import messagebox, simpledialog, ttk
from datetime import datetime
import os
import random # للتحقق بخطوتين
import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg

# --- الألوان والهوية البصرية ---
BG_MAIN = "#0f172a"      # Deep Navy
BG_CARD = "#1e293b"      # Slate Blue
ACCENT = "#3b82f6"       # Bright Blue
SUCCESS = "#10b981"      # Emerald
WARNING = "#f59e0b"      # Amber
DANGER = "#ef4444"       # Rose
TEXT_PRIMARY = "#f8fafc"

# --- محرك الأمان (التشفير) ---
def encrypt(text):
    return "".join(chr(ord(c) + 3) for c in str(text))

def decrypt(text):
    return "".join(chr(ord(c) - 3) for c in str(text))

class EnterpriseBankSystem:
    def __init__(self, root):
        self.root = root
        self.root.title("AL-QADRI ENTERPRISE BANKING SOLUTION")
        self.root.geometry("700x850")
        self.root.configure(bg=BG_MAIN)
        self.db_file = "vault_master.dat"
        self.backup_file = "vault_backup.dat"
        self.log_file = "audit_trail.txt" # سجل العمليات (Statement)
        
        self.show_login_page()

    # 1. نظام الدخول والتحقق بخطوتين (2FA Simulation)
    def show_login_page(self):
        self.login_frame = tk.Frame(self.root, bg=BG_MAIN)
        self.login_frame.pack(expand=True, fill="both")

        tk.Label(self.login_frame, text="🏛️ بوابة الدخول الآمن", font=("Helvetica", 24, "bold"), bg=BG_MAIN, fg=SUCCESS).pack(pady=40)
        
        tk.Label(self.login_frame, text="اسم المستخدم:", bg=BG_MAIN, fg=TEXT_PRIMARY).pack()
        self.u_ent = tk.Entry(self.login_frame, font=("Arial", 12), justify="center"); self.u_ent.pack(pady=10)

        tk.Label(self.login_frame, text="كلمة المرور:", bg=BG_MAIN, fg=TEXT_PRIMARY).pack()
        self.p_ent = tk.Entry(self.login_frame, show="*", font=("Arial", 12), justify="center"); self.p_ent.pack(pady=10)

        tk.Button(self.login_frame, text="طلب رمز التحقق (OTP)", bg=ACCENT, fg="white", command=self.send_otp).pack(pady=20)

    def send_otp(self):
        if self.u_ent.get() == "admin" and self.p_ent.get() == "1234":
            self.current_otp = str(random.randint(1000, 9999))
            # محاكاة إرسال بريد
            messagebox.showinfo("2FA System", f"📧 تم إرسال رمز التحقق إلى بريدك الإلكتروني:\n\nCODE: {self.current_otp}")
            
            otp_in = simpledialog.askstring("التحقق بخطوتين", "أدخل الرمز المكون من 4 أرقام:")
            if otp_in == self.current_otp:
                self.login_frame.destroy()
                self.main_hub()
                self.auto_backup() # نسخ احتياطي عند الدخول
            else:
                messagebox.showerror("خطأ", "رمز التحقق غير صحيح!")
        else:
            messagebox.showerror("خطأ", "بيانات الدخول خاطئة!")

    # 2. لوحة التحكم الرئيسية (Hub)
    def main_hub(self):
        nav = tk.Frame(self.root, bg=BG_CARD, pady=20)
        nav.pack(fill="x")
        tk.Label(nav, text="نظام الإدارة المصرفية المتكامل", font=("Tajawal", 18), bg=BG_CARD, fg=TEXT_PRIMARY).pack()

        main_area = tk.Frame(self.root, bg=BG_MAIN)
        main_area.pack(pady=30)

        actions = [
            ("👤 فتح حساب (تصنيف آلي)", SUCCESS, self.new_account),
            ("💸 تحويل أموال بين الحسابات", ACCENT, self.transfer_money),
            ("💰 إيداع / 🏧 سحب", WARNING, self.transaction_ui),
            ("❄️ إدارة تجميد الحسابات", DANGER, self.freeze_ui),
            ("📊 ذكاء الأعمال (BI Analytics)", "#8b5cf6", self.show_bi_dashboard),
            ("📜 سجل العمليات (Audit Log)", "#64748b", self.show_audit_logs),
            ("💾 إنشاء نسخة احتياطية يدويًا", "#10b981", self.manual_backup),
        ]

        for txt, clr, cmd in actions:
            tk.Button(main_area, text=txt, bg=clr, fg="white", font=("Tajawal", 11, "bold"), 
                      width=40, height=2, bd=0, command=cmd).pack(pady=6)

    # 3. ميزة فتح الحساب مع التصنيف الآلي (Tiering)
    def new_account(self):
        win = tk.Toplevel(self.root); win.geometry("350x450"); win.configure(bg=BG_CARD)
        
        tk.Label(win, text="إضافة عميل جديد", bg=BG_CARD, fg=SUCCESS, font=(14)).pack(pady=20)
        entries = {}
        for f in ["رقم الحساب", "اسم العميل", "الإيداع الأولي"]:
            tk.Label(win, text=f, bg=BG_CARD, fg="white").pack()
            e = tk.Entry(win, justify="center"); e.pack(pady=5); entries[f] = e

        def save():
            acc, name, bal = entries["رقم الحساب"].get(), entries["اسم العميل"].get(), float(entries["الإيداع الأولي"].get())
            status = "Active" # نشط افتراضياً
            # التصنيف الآلي
            tier = "VIP" if bal >= 10000 else "Gold" if bal >= 5000 else "Standard"
            
            data = f"{acc}|{name}|{bal}|{status}|{tier}"
            with open(self.db_file, "a", encoding="utf-8") as f:
                f.write(encrypt(data) + "\n")
            
            self.log_action(f"فتح حساب جديد: {acc} - تصنيف: {tier}")
            messagebox.showinfo("تم", f"تم تسجيل العميل بتصنيف: {tier}\n📧 تم إرسال بريد ترحيبي!")
            win.destroy()

        tk.Button(win, text="حفظ وتصنيف", bg=SUCCESS, command=save).pack(pady=20)

    # 4. نظام تحويل الأموال وتجميد الحسابات
    def transfer_money(self):
        from_acc = simpledialog.askstring("التحويل", "رقم حساب المرسل:")
        to_acc = simpledialog.askstring("التحويل", "رقم حساب المستلم:")
        amount = simpledialog.askfloat("التحويل", "المبلغ المراد تحويله:")
        
        if self.update_balance(from_acc, -amount, is_transfer=True) and self.update_balance(to_acc, amount, is_transfer=True):
            self.log_action(f"تحويل: من {from_acc} إلى {to_acc} بمبلغ {amount}$")
            messagebox.showinfo("نجاح", "تمت عملية التحويل بنجاح! 📧 تم إرسال إشعار للطرفين.")
        else:
            messagebox.showerror("فشل", "تأكد من الأرقام، كفاية الرصيد، أو أن الحساب غير مجمد!")

    def freeze_ui(self):
        acc = simpledialog.askstring("الأمن", "أدخل رقم الحساب للتجميد/التنشيط:")
        self.toggle_freeze(acc)

    def toggle_freeze(self, acc_id):
        all_data, found = [], False
        with open(self.db_file, "r", encoding="utf-8") as f:
            for line in f:
                p = decrypt(line.strip()).split("|")
                if p[0] == acc_id:
                    p[3] = "Frozen" if p[3] == "Active" else "Active"
                    found = True
                    state = p[3]
                all_data.append(encrypt("|".join(p)))
        if found:
            with open(self.db_file, "w", encoding="utf-8") as f:
                for l in all_data: f.write(l + "\n")
            self.log_action(f"تغيير حالة الحساب {acc_id} إلى {state}")
            messagebox.showinfo("نظام الأمن", f"تم تغيير حالة الحساب إلى: {state}")

    # 5. دوال الإيداع والسحب مع سجل العمليات
    def transaction_ui(self):
        acc = simpledialog.askstring("العمليات", "رقم الحساب:")
        mode = simpledialog.askstring("العمليات", "اكتب (سحب) أو (إيداع):")
        amt = simpledialog.askfloat("العمليات", "المبلغ:")
        
        change = amt if mode == "إيداع" else -amt
        if self.update_balance(acc, change):
            self.log_action(f"عملية {mode}: للحساب {acc} بمبلغ {amt}$")
            messagebox.showinfo("إيصال", f"تمت العملية! الرصيد المحدث سجل في النظام.")
        else:
            messagebox.showerror("خطأ", "فشلت العملية. راجع حالة الحساب والرصيد.")

    def update_balance(self, acc_id, change, is_transfer=False):
        if not os.path.exists(self.db_file): return False
        all_data, found = [], False
        with open(self.db_file, "r", encoding="utf-8") as f:
            for line in f:
                p = decrypt(line.strip()).split("|")
                if p[0] == acc_id:
                    if p[3] == "Frozen": return False # الحساب مجمد
                    new_bal = float(p[2]) + change
                    if new_bal < 0: return False # رصيد غير كافٍ
                    p[2] = str(new_bal)
                    # تحديث التصنيف بعد الحركة المالية
                    p[4] = "VIP" if new_bal >= 10000 else "Gold" if new_bal >= 5000 else "Standard"
                    found = True
                all_data.append(encrypt("|".join(p)))
        if found:
            with open(self.db_file, "w", encoding="utf-8") as f:
                for l in all_data: f.write(l + "\n")
        return found

    # 6. سجل العمليات والنسخ الاحتياطي (MIS Audit & Continuity)
    def log_action(self, msg):
        with open(self.log_file, "a", encoding="utf-8") as f:
            f.write(f"[{datetime.now()}] {msg}\n")

    def auto_backup(self):
        if os.path.exists(self.db_file):
            with open(self.db_file, "r") as f, open(self.backup_file, "w") as b:
                b.write(f.read())

    def manual_backup(self):
        self.auto_backup()
        messagebox.showinfo("النسخ الاحتياطي", "✅ تم إنشاء نسخة احتياطية مشفرة بنجاح (vault_backup.dat)")

    def show_audit_logs(self):
        win = tk.Toplevel(self.root); win.title("Audit Trail")
        txt = tk.Text(win, bg="black", fg=SUCCESS)
        txt.pack(fill="both", expand=True)
        if os.path.exists(self.log_file):
            with open(self.log_file, "r", encoding="utf-8") as f:
                txt.insert("1.0", f.read())

    # 7. لوحة تحليل ذكاء الأعمال (BI Dashboard)
    def show_bi_dashboard(self):
        if not os.path.exists(self.db_file): return
        
        tiers = {"VIP": 0, "Gold": 0, "Standard": 0}
        balances = []
        with open(self.db_file, "r", encoding="utf-8") as f:
            for line in f:
                p = decrypt(line.strip()).split("|")
                tiers[p[4]] += 1
                balances.append(float(p[2]))

        win = tk.Toplevel(self.root); win.geometry("800x600"); win.configure(bg="white")
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))

        # رسم توزيع التصنيفات (Pie Chart)
        ax1.pie(tiers.values(), labels=tiers.keys(), autopct='%1.1f%%', colors=[SUCCESS, WARNING, ACCENT])
        ax1.set_title("توزيع تصنيفات العملاء")

        # رسم إجمالي السيولة (Bar)
        ax2.bar(["إجمالي السيولة"], [sum(balances)], color=ACCENT)
        ax2.set_title(f"إجمالي المبالغ: {sum(balances)}$")

        canvas = FigureCanvasTkAgg(fig, master=win)
        canvas.draw(); canvas.get_tk_widget().pack(fill="both", expand=True)

if __name__ == "__main__":
    root = tk.Tk()
    app = EnterpriseBankSystem(root)
    root.mainloop()

